# Lab 2: Operator learning on point clouds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maximilianherde/ricam-foundation-models-for-pdes/blob/main/notebooks/02_gaot_point_clouds.ipynb)

**RICAM Training School, Linz, 8 October 2026**

Lab 1 worked on a uniform grid. Data from engineering applications usually
comes on meshes or point clouds, with varying numbers of points and no FFT. In
this lab you

- take the Allen–Cahn data from Lab 1 and keep only the values at scattered
  points;
- write the kernel integral as a sum over neighbouring points;
- build a small GAOT: a geometry-aware encoder, a transformer on a latent grid,
  and a decoder that can be evaluated at any point;
- evaluate it at points other than the ones it was given.

Set the Colab runtime to a GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
torch.manual_seed(0)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

N_TRAIN, N_TEST = 128, 240       # trajectories
SNAPSHOTS = slice(0, 15, 2)      # snapshots 0, 2, ..., 14, as in Lab 1
RES = 128

## The data

The same data as in Lab 1: the Allen–Cahn dataset of the Poseidon paper,

$$u_t = \Delta u - \varepsilon^2 u\,(u^2-1), \qquad \varepsilon = 220,$$

on the unit square, 20 snapshots on a
$128\times128$ grid. Training data are the first 128 trajectories of
`solution_0.nc`, the test set is the paper's: the last 240 trajectories of
`solution_3.nc`.

**The task** is the one-step map of Lab 1: as in the paper we use the
snapshots $0, 2, \dots, 14$, numbered $k = 0, \dots, 7$, and predict $u_{k+1}$
from $u_k$. Each trajectory gives 7 pairs. Since the task is the same, the
errors can be compared with the FNO of Lab 1.

Fields are standardised with the dataset mean and standard deviation, as in
Poseidon, and errors are reported on the standardised fields. They could
equally be reported on the physical field; the values differ slightly.

In [ ]:
try:
    import fsspec, h5py
except ImportError:
    !pip install -q fsspec h5py
    import fsspec, h5py

URL = "https://huggingface.co/datasets/camlab-ethz/ACE/resolve/main/solution_{}.nc"
MEAN, STD = 0.002484262, 0.65351176           # Poseidon's constants for ACE


def load_ace(file, start, stop, snapshots=slice(None), cache="ace_cache"):
    tag = f"{snapshots.start}_{snapshots.stop}_{snapshots.step}"
    path = os.path.join(cache, f"ace{file}_{start}_{stop}_{tag}.npy")
    if os.path.exists(path):
        return np.load(path)
    os.makedirs(cache, exist_ok=True)
    with fsspec.open(URL.format(file), "rb", block_size=2**22) as f, h5py.File(f, "r") as h:
        data = h["solution"][start:stop, snapshots]
    np.save(path, data)
    return data


train = torch.from_numpy((load_ace(0, 0, N_TRAIN, SNAPSHOTS) - MEAN) / STD).to(device)
test = torch.from_numpy((load_ace(3, 3750 - N_TEST, 3750, SNAPSHOTS) - MEAN) / STD).to(device)

train_in, train_out = train[:, :-1].reshape(-1, RES, RES), train[:, 1:].reshape(-1, RES, RES)
test_in, test_out = test[:, :-1].reshape(-1, RES, RES), test[:, 1:].reshape(-1, RES, RES)
print(f"{len(train_in)} training pairs, {len(test_in)} test pairs")

## From a grid to a point cloud

A point cloud is a set of coordinates with values attached. We draw random
points in the unit square and read the field there by bilinear interpolation
(`grid_sample`).

In [ ]:
def sample_points(n_points):
    # uniformly random coordinates in [0,1]^2
    return torch.rand(n_points, 2, device=device)


def read_field(fields, coords):
    # fields (B, H, W), coords (P, 2) -> values (B, P).
    # coords[:, 0] runs along the last axis of the field, coords[:, 1] along the first.
    grid = (coords * 2 - 1).view(1, 1, -1, 2).expand(fields.shape[0], -1, -1, -1)
    out = F.grid_sample(fields.unsqueeze(1), grid, mode="bilinear",
                        padding_mode="reflection", align_corners=False)
    return out[:, 0, 0]


pts = sample_points(3000)
vals = read_field(test_out[:1], pts)[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
axes[0].imshow(test_out[0].cpu(), origin="lower", extent=(0, 1, 0, 1),
               cmap="gist_ncar", vmin=-1.6, vmax=1.6)
axes[0].set_title("on the grid")
axes[1].scatter(pts[:, 0].cpu(), pts[:, 1].cpu(), c=vals.cpu(), s=5,
                cmap="gist_ncar", vmin=-1.6, vmax=1.6)
axes[1].set_title("at 3000 random points")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

## The kernel integral on a point cloud

A neural operator layer has the form

$$v_{\ell+1}(x) = \sigma\Big(W v_\ell(x) + \int_D \kappa(x,y)\, v_\ell(y)\, dy\Big).$$

On a grid the integral can be computed with a convolution or an FFT. On a
point cloud GAOT approximates it by a sum over the points within a radius $r$
of $x$:

$$\int_D \kappa(x,y)\,v(y)\,dy \;\approx\; \sum_{y \in B_r(x)} \alpha(x,y)\, k(x,y)\, v(y).$$

- $k(x,y)$ is a small MLP of the coordinates of $x$ and $y$;
- $\alpha(x,y)$ are attention weights: a softmax over the neighbourhood $B_r(x)$
  of the cosine similarity between the coordinates of $x$ and $y$, so they sum
  to one over each neighbourhood.

The official code finds the neighbourhoods with `torch_cluster`; here we
write the search ourselves.

### Exercise 1

Fill in `radius_neighbors`: for query points `[Q, 2]` and source points
`[S, 2]`, return two index tensors `q, s` that list every pair with
$|$`source[s]` $-$ `query[q]`$| \le r$, one entry per pair.

<details><summary><b>Hint</b></summary>

Compute all pairwise distances at once (`torch.cdist`). The indices of the
`True` entries of a boolean matrix are given by `.nonzero(as_tuple=True)`.

</details>

In [ ]:
def radius_neighbors(query, source, r):
    # returns (q [E], s [E]): all pairs with |source[s] - query[q]| <= r
    raise NotImplementedError("Exercise 1")

In [ ]:
# Check
_query, _source = torch.rand(5, 2, device=device), torch.rand(200, 2, device=device)
_q, _s = radius_neighbors(_query, _source, 0.2)
_d = (_query[:, None] - _source[None]).norm(dim=-1)
assert len(_q) == (_d <= 0.2).sum() and (_d[_q, _s] <= 0.2).all()
print(f"ok, {len(_q)} pairs")

## MAGNO

GAOT's layer (`src/model/layers/magno.py` in the official code) maps values on
source points to values on query points:

1. find the radius neighbourhood of every query point;
2. compute the attention-weighted kernel sum above;
3. compute a **geometric embedding** of each neighbourhood: the number of
   neighbours, the mean and variance of their distances to $x$, the offset of
   their centroid from $x$, and the eigenvalues of their covariance. These are
   standardised over the query points and passed through an MLP;
4. concatenate the result of 2 and 3 and map it back with a linear layer.

The encoder first lifts the input values with a linear layer; the decoder ends
with a linear projection to the output. GAOT can also use several radii and
average the results; its default, which we use, is a single radius.

The layer assumes no grid, no fixed number of points and no ordering, so the
same layer maps points to a latent grid (encoder) and the latent grid to
points (decoder). Sums over neighbourhoods are done with `index_add_`.

In [ ]:
def mlp(sizes, act=nn.GELU):
    layers = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), act()]
    return nn.Sequential(*layers[:-1])                 # no activation after the last layer


def segment_softmax(scores, q, n_queries):
    # softmax of scores [E] over the edges that share the same query point q [E]
    m = torch.full((n_queries,), -torch.inf, device=scores.device).scatter_reduce(
        0, q, scores, reduce="amax")
    e = (scores - m[q]).exp()
    return e / torch.zeros(n_queries, device=scores.device).index_add(0, q, e)[q]


def neighbourhood_stats(query, source, q, s):
    # per query point: [number of neighbours, mean distance, distance variance,
    # centroid offset (2), covariance eigenvalues (2)], standardised over the queries
    n_q = len(query)
    count = torch.zeros(n_q, device=query.device).index_add(0, q, torch.ones_like(q, dtype=query.dtype))
    norm = count.clamp(min=1)

    def mean(v):                                      # mean over each neighbourhood
        out = torch.zeros(n_q, *v.shape[1:], device=v.device).index_add(0, q, v)
        return out / norm.view(-1, *[1] * (v.ndim - 1))

    dist = (source[s] - query[q]).norm(dim=-1)
    d_avg = mean(dist)
    d_var = (mean(dist**2) - d_avg**2).clamp(min=0)
    centroid = mean(source[s])
    c = source[s] - centroid[q]
    cov = mean(c[:, :, None] * c[:, None, :])          # [Q, 2, 2]
    a, b, d = cov[:, 0, 0], cov[:, 0, 1], cov[:, 1, 1]
    root = (((a - d) / 2) ** 2 + b**2).sqrt()
    eig = torch.stack([(a + d) / 2 + root, (a + d) / 2 - root], dim=1)
    feats = torch.cat([count[:, None], d_avg[:, None], d_var[:, None],
                       centroid - query, eig], dim=1)
    feats[count == 0] = 0
    std = feats.std(0, keepdim=True)
    return (feats - feats.mean(0, keepdim=True)) / torch.where(std < 1e-6, 1.0, std)


class MAGNO(nn.Module):
    # values on `source` points -> values on `query` points (GAOT's MAGNO, one radius)
    def __init__(self, in_ch, out_ch, radius, hidden=64):
        super().__init__()
        self.radius = radius
        C = max(in_ch, out_ch)
        self.lift = nn.Linear(in_ch, C) if in_ch != C else nn.Identity()      # encoder
        self.kernel = mlp([4, hidden, hidden, hidden, C])                      # k(x, y)
        self.geo = nn.Sequential(nn.Linear(7, 64), nn.ReLU(), nn.Linear(64, C), nn.ReLU())
        self.recovery = nn.Linear(2 * C, C)
        self.project = nn.Linear(C, out_ch) if out_ch != C else nn.Identity()  # decoder

    def forward(self, query_coords, source_coords, source_values):
        # source_values: [B, S, in_ch]; coordinates are shared across the batch
        f = self.lift(source_values)                                           # [B, S, C]
        q, s = radius_neighbors(query_coords, source_coords, self.radius)
        x, y = query_coords[q], source_coords[s]                               # [E, 2]
        k = self.kernel(torch.cat([y, x], dim=-1))                             # [E, C]
        alpha = segment_softmax(F.cosine_similarity(x, y, dim=-1), q, len(query_coords))
        msg = f[:, s] * (alpha[:, None] * k)                                   # [B, E, C]
        out = torch.zeros(f.shape[0], len(query_coords), f.shape[2], device=f.device)
        out = out.index_add(1, q, msg)                                         # [B, Q, C]
        geo = self.geo(neighbourhood_stats(query_coords, source_coords, q, s))
        out = self.recovery(torch.cat([out, geo.expand(out.shape[0], -1, -1)], dim=-1))
        return self.project(out)                                               # [B, Q, out_ch]

## The model

```
point cloud --MAGNO--> 64x64 latent tokens --2x2 patches--> transformer --> tokens --MAGNO--> query points
```

The geometry is handled only by the encoder and the decoder. The transformer
works on a fixed grid of tokens, where attention is cheap to batch. This is the
main design choice of GAOT.

As in GAOT, the transformer does not see the latent tokens one by one. Each
$2\times2$ block of latent tokens is concatenated into one patch token
(`patchify`), so attention runs over $32\times32 = 1024$ patches instead of
4096 tokens. A linear layer maps each patch to the transformer width, and
another maps it back before `unpatchify`.

The latent grid and the radius are GAOT's defaults: $64\times64$ latent
tokens, $2\times2$ patches, and $r = 0.033$, about two latent grid spacings.

### Exercise 2

Fill in `forward`: encode the input values onto the latent grid, process the
latent grid with the transformer on $2\times2$ patches, and decode at the query
points. All modules are defined in `__init__`; `patchify` and `unpatchify` are
given.

<details><summary><b>Hint</b></summary>

The order is: `encoder` (from the input points to `latent_coords`) →
`patchify` → `patch_in` → add `pos` → `processor` → `patch_out` →
`unpatchify` → `decoder` (from `latent_coords` to the query points).

</details>

In [ ]:
class MiniGAOT(nn.Module):
    def __init__(self, latent=64, dim=64, patch=2, hidden=256, depth=4, heads=4,
                 radius=0.033):
        super().__init__()
        self.latent, self.patch = latent, patch
        g = torch.linspace(0, 1, latent + 1)[:-1] + 0.5 / latent
        yy, xx = torch.meshgrid(g, g, indexing="ij")
        self.register_buffer("latent_coords", torch.stack([xx.reshape(-1), yy.reshape(-1)], 1))
        self.encoder = MAGNO(1, dim, radius)
        self.patch_in = nn.Linear(patch * patch * dim, hidden)
        self.pos = nn.Parameter(torch.zeros(1, (latent // patch) ** 2, hidden))
        layer = nn.TransformerEncoderLayer(hidden, heads, 4 * hidden, batch_first=True,
                                           norm_first=True, dropout=0.0)
        self.processor = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.patch_out = nn.Linear(hidden, patch * patch * dim)
        self.decoder = MAGNO(dim, 1, radius)

    def patchify(self, t):
        # [B, L*L, C] -> [B, (L/P)^2, P*P*C]: each P x P block of latent tokens becomes one token
        B, _, C = t.shape
        L, P = self.latent, self.patch
        t = t.view(B, L // P, P, L // P, P, C).permute(0, 1, 3, 2, 4, 5)
        return t.reshape(B, (L // P) ** 2, P * P * C)

    def unpatchify(self, t):
        # inverse of patchify
        B = t.shape[0]
        L, P = self.latent, self.patch
        C = t.shape[-1] // (P * P)
        t = t.view(B, L // P, L // P, P, P, C).permute(0, 1, 3, 2, 4, 5)
        return t.reshape(B, L * L, C)

    def forward(self, coords, values, query_coords):
        # values [B, P, 1] at coords -> [B, Q, 1] at query_coords
        raise NotImplementedError("Exercise 2")

In [ ]:
# Check: the query points can differ from the input points
_m = MiniGAOT(latent=8, dim=32, depth=2).to(device)
_y = _m(sample_points(200), torch.randn(2, 200, 1, device=device), sample_points(37))
assert _y.shape == (2, 37, 1)
print("ok")

## Training

Each batch uses a new random point cloud: the input field and the target field
are both read at these points. The model never sees the same set of points
twice. The loss is the relative $L^1$ error.

In [ ]:
LATENT, DIM, DEPTH = 64, 64, 4
STEPS, BATCH, POINTS = 1500, 16, 3000



class LiveLoss:
    # Plots the training loss while training. Call update(loss) every step and
    # close() at the end; the plot shows the mean over every `every` steps.
    def __init__(self, title="", every=20):
        self.every, self.steps, self.values = every, [], []
        self.window, self.n = 0.0, 0
        self.fig, self.ax = plt.subplots(figsize=(5, 2.6))
        self.title = title
        self.handle = display(self.fig, display_id=True)
        plt.close(self.fig)

    def update(self, loss):
        self.window = self.window + loss.detach()
        self.n += 1
        if self.n % self.every == 0:
            self.steps.append(self.n)
            self.values.append(self.window.item() / self.every)
            self.window = 0.0
            self.draw()

    def draw(self):
        self.ax.clear()
        self.ax.semilogy(self.steps, self.values)
        self.ax.set_xlabel("step"); self.ax.set_ylabel("training loss")
        self.ax.set_title(self.title, fontsize=9)
        self.fig.tight_layout()
        self.handle.update(self.fig)

    def close(self):
        if self.steps:
            self.draw()



def rel_l1(pred, true):
    # one number per sample
    return (pred - true).abs().sum(dim=(1, 2)) / true.abs().sum(dim=(1, 2))


def make_batch(n_points):
    idx = torch.randint(0, len(train_in), (BATCH,), device=device)
    coords = sample_points(n_points)
    x = read_field(train_in[idx], coords).unsqueeze(-1)       # [B, P, 1]
    y = read_field(train_out[idx], coords).unsqueeze(-1)
    return coords, x, y


model = MiniGAOT(latent=LATENT, dim=DIM, depth=DEPTH).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)        # lr decays from 2e-3 to 0
live = LiveLoss("MiniGAOT")
for step in tqdm(range(STEPS)):
    coords, x, y = make_batch(POINTS)
    loss = rel_l1(model(coords, x, coords), y).mean()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    live.update(loss)
live.close()

## Querying at other points

The decoder can be evaluated at any coordinates, not only at the input points.
We give the model each test input at 3000 random points, as in training, and
query the prediction at other numbers of random points.

In [ ]:
@torch.no_grad()
def evaluate(n_query, reps=1, batch=16):
    model.eval()
    errs = []
    for _ in range(reps):
        coords = sample_points(POINTS)                 # input points
        query = sample_points(n_query)                 # output points
        for i in range(0, len(test_in), batch):
            x = read_field(test_in[i:i + batch], coords).unsqueeze(-1)
            y = read_field(test_out[i:i + batch], query).unsqueeze(-1)
            errs.append(rel_l1(model(coords, x, query), y))
    return torch.cat(errs).median().item()


print(f"input: {POINTS} uniform points\n")
print(f"{'query points':>12}  {'median rel. L1':>15}")
for n in [750, 1500, 3000, 6000, 12000]:
    print(f"{n:12d}  {evaluate(n):15.4f}")

Finally, all $128\times128$ grid points as queries:

In [ ]:
@torch.no_grad()
def predict_on_grid(j=0):
    model.eval()
    coords = sample_points(POINTS)
    x = read_field(test_in[j:j + 1], coords).unsqueeze(-1)
    g = (torch.arange(RES, device=device) + 0.5) / RES                  # pixel centres
    yy, xx = torch.meshgrid(g, g, indexing="ij")
    query = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    return coords, model(coords, x, query).reshape(RES, RES)


coords, pred = predict_on_grid()
truth = test_out[0]
err = (pred - truth).abs()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].scatter(coords[:, 0].cpu(), coords[:, 1].cpu(), s=3, c="k")
axes[0].set_title(f"input: {POINTS} points")
axes[0].set_aspect("equal"); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
for ax, f, t, lim in ((axes[1], pred, "prediction on the grid", (-1.6, 1.6)),
                       (axes[2], truth, "true", (-1.6, 1.6)),
                       (axes[3], err, "|error|", (0, float(err.max())))):
    ax.imshow(f.cpu(), origin="lower", extent=(0, 1, 0, 1), cmap="gist_ncar", vmin=lim[0], vmax=lim[1])
    ax.set_title(t)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(f"relative L1 on the grid: {(err.sum() / truth.abs().sum()).item():.4f}")

### Rollout

For the last test trajectory, we start from $u_0$ at 3000 random points and
apply the model seven times to reach $u_7$ (snapshot 14). The model predicts
the next field at the same 3000 points, and this prediction is the input of
the next step. Errors are measured at these points. Shown in physical units:

In [ ]:
traj = test[-1]                                          # (8, RES, RES), standardised
coords = sample_points(POINTS)
true = read_field(traj, coords)                          # (8, P): u_k at the points

x, preds = true[:1, :, None], [true[0]]
with torch.no_grad():
    model.eval()
    for k in range(1, 8):
        x = model(coords, x, coords)                     # u_k at the points, next input
        preds.append(x[0, :, 0])
pred = torch.stack(preds)
err = [rel_l1(pred[k][None, :, None], true[k][None, :, None]).item() for k in range(1, 8)]
print("rollout error per step:", " ".join(f"{e:.3f}" for e in err))

true_u = (true * STD + MEAN).cpu().numpy()
pred_u = (pred * STD + MEAN).cpu().numpy()
fields = [true_u, pred_u, np.abs(pred_u - true_u)]
xy = coords.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(8.1, 2.9))
dots = []
for ax, f, t, (lo, hi) in zip(axes, fields, ["true", "GAOT rollout", "|error|"],
                              [(-1, 1), (-1, 1), (0, 0.5)]):
    dots.append(ax.scatter(xy[:, 0], xy[:, 1], c=f[0], s=4, cmap="gist_ncar", vmin=lo, vmax=hi))
    ax.set_title(t, fontsize=9); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
label = fig.suptitle("k = 0  (snapshot 0)")


def frame(k):
    for d, f in zip(dots, fields):
        d.set_array(f[k])
    label.set_text(f"k = {k}  (snapshot {2 * k})")
    return dots


anim = animation.FuncAnimation(fig, frame, frames=8, interval=400)
plt.close(fig)
HTML(anim.to_jshtml())

## What is missing compared with GAOT

The encoder and decoder follow GAOT's MAGNO with its default settings (one
radius, cosine attention, statistical geometric embedding), and the latent grid
is patched as in GAOT. Differences to the published model:

- the transformer is PyTorch's standard encoder layer; GAOT uses its own block
  with RMSNorm and grouped-query attention;
- one radius, where GAOT can combine several;
- no edge dropping and no caching of neighbourhoods, which the official code
  uses for large point clouds;
- 2D only, with a regular latent grid; GAOT-3D adds other tokenisations.

Wen et al., *Geometry Aware Operator Transformer*, NeurIPS 2025. The code is
at `github.com/camlab-ethz/GAOT`.